In [6]:
# Board composition measured as at 31 October each year
# Source: FTSE Women Leaders Review Portal
# Note: timing differs from fiscal year-end financials — discuss in methodology

In [1]:
import pandas as pd
import numpy as np
import os

# create folders if they don't exist
for folder in ["data/raw", "data/processed", "outputs/figures", "outputs/tables"]:
    os.makedirs(folder, exist_ok=True)

print("Folders ready")
print("pandas:", pd.__version__)

Folders ready
pandas: 2.2.3


In [2]:
url = "https://ftsewomenleaders.com/company-rankings/"

tables = pd.read_html(url)

print(f"Number of tables found: {len(tables)}")
for i, t in enumerate(tables):
    print(f"\nTable {i}: shape {t.shape}")
    print("Columns:", t.columns.tolist())

Number of tables found: 3

Table 0: shape (95, 8)
Columns: ['Rank', 'Company', 'Supersector', 'Women on Boards', 'Board Size', 'Total Women on Boards', 'Executive Women on Boards', 'Combined Exec.Comm & DRs']

Table 1: shape (165, 8)
Columns: ['Rank', 'Company', 'Supersector', 'Women on Boards', 'Board Size', 'Total Women on Boards', 'Executive Women on Boards', 'Combined Exec.Comm & DRs']

Table 2: shape (48, 8)
Columns: ['Rank', 'Company', 'Supersector', 'Women on Boards', 'Board Size', 'Total Women on Boards', 'Executive Women on Boards', 'Combined Exec.Comm & DRs']


In [3]:
ftse100 = tables[0]

print("Shape:", ftse100.shape)
print("\nData types:")
print(ftse100.dtypes)
print("\nFirst 10 rows:")
ftse100.head(10)

Shape: (95, 8)

Data types:
Rank                          int64
Company                      object
Supersector                  object
Women on Boards              object
Board Size                    int64
Total Women on Boards         int64
Executive Women on Boards     int64
Combined Exec.Comm & DRs     object
dtype: object

First 10 rows:


,Rank,Company,Supersector,Women on Boards,Board Size,Total Women on Boards,Executive Women on Boards,Combined Exec.Comm & DRs
0,1,Burberry Group Plc,Consumer Products & Services,55.6%,9,5,1,56.4%
1,2,Next Plc,Retail,33.3%,12,4,1,53.6%
2,3,Marks & Spencer Group Plc,"Personal Care, Drug & Grocery Stores",66.7%,9,6,1,50.0%
3,4,Phoenix Group Holdings Plc,Insurance,58.3%,12,7,0,49.5%
4,5,Convatec Group Plc,Health Care,55.6%,9,5,1,49.4%
5,6,Diageo Plc,"Food, Beverage & Tobacco",77.8%,9,7,1,49.1%
6,7,Haleon Plc,Health Care,58.3%,12,7,1,47.8%
7,8,Pearson Plc,Media,63.6%,11,7,1,47.6%
8,9,AstraZeneca Plc,Health Care,50.0%,14,7,1,46.8%
9,10,National Grid Plc,Utilities,41.7%,12,5,1,45.7%


In [4]:
print(ftse100.tail(5))
print("\nMissing values per column:")
print(ftse100.isnull().sum())

    Rank                   Company                   Supersector  \
90    91     JD Sports Fashion Plc                        Retail   
91    92        Intertek Group Plc   Industrial Goods & Services   
92    93           Antofagasta Plc               Basic Resources   
93    94         Ashtead Group Plc   Industrial Goods & Services   
94    95  Games Workshop Group Plc  Consumer Products & Services   

   Women on Boards  Board Size  Total Women on Boards  \
90           30.0%          10                      3   
91           30.8%          13                      4   
92           30.8%          13                      4   
93           22.2%           9                      2   
94           28.6%           7                      2   

    Executive Women on Boards Combined Exec.Comm & DRs  
90                          0                    32.6%  
91                          0                    27.7%  
92                          0                    24.7%  
93                  

In [5]:
df = ftse100.copy()

# Convert percentage strings to floats
df["WomenPct"] = df["Women on Boards"].str.replace("%", "", regex=False).astype(float)
df["ExecCommPct"] = df["Combined Exec.Comm & DRs"].str.replace("%", "", regex=False).astype(float)

# Rename to clean column names
df = df.rename(columns={
    "Board Size": "BoardSize",
    "Total Women on Boards": "TotalWomen",
    "Executive Women on Boards": "ExecWomen"
})

# Tag the year — this is the 2026 report, reporting on 2025 data
df["Year"] = 2025

# Standardised company key for later merging
df["CompanyKey"] = (df["Company"]
                    .str.upper()
                    .str.replace(r"\b(PLC|LIMITED|LTD|GROUP|HOLDINGS)\b", "", regex=True)
                    .str.replace(r"[^A-Z0-9 ]", "", regex=True)
                    .str.replace(r"\s+", " ", regex=True)
                    .str.strip())

keep = ["CompanyKey", "Company", "Year", "Supersector", "WomenPct", 
        "BoardSize", "TotalWomen", "ExecWomen", "ExecCommPct"]
df_clean = df[keep]

print(df_clean.dtypes)
print()
df_clean.head(10)

CompanyKey      object
Company         object
Year             int64
Supersector     object
WomenPct       float64
BoardSize        int64
TotalWomen       int64
ExecWomen        int64
ExecCommPct    float64
dtype: object



,CompanyKey,Company,Year,Supersector,WomenPct,BoardSize,TotalWomen,ExecWomen,ExecCommPct
0,BURBERRY,Burberry Group Plc,2025,Consumer Products & Services,55.6,9,5,1,56.4
1,NEXT,Next Plc,2025,Retail,33.3,12,4,1,53.6
2,MARKS SPENCER,Marks & Spencer Group Plc,2025,"Personal Care, Drug & Grocery Stores",66.7,9,6,1,50.0
3,PHOENIX,Phoenix Group Holdings Plc,2025,Insurance,58.3,12,7,0,49.5
4,CONVATEC,Convatec Group Plc,2025,Health Care,55.6,9,5,1,49.4
5,DIAGEO,Diageo Plc,2025,"Food, Beverage & Tobacco",77.8,9,7,1,49.1
6,HALEON,Haleon Plc,2025,Health Care,58.3,12,7,1,47.8
7,PEARSON,Pearson Plc,2025,Media,63.6,11,7,1,47.6
8,ASTRAZENECA,AstraZeneca Plc,2025,Health Care,50.0,14,7,1,46.8
9,NATIONAL GRID,National Grid Plc,2025,Utilities,41.7,12,5,1,45.7


In [7]:
print(df_clean["WomenPct"].describe())
print("\nFirms below 30% (critical mass threshold):", (df_clean["WomenPct"] < 30).sum())
print("Firms at or above 30%:", (df_clean["WomenPct"] >= 30).sum())
print("\nBoard size range:", df_clean["BoardSize"].min(), "to", df_clean["BoardSize"].max())

count    95.000000
mean     44.510526
std       9.582603
min      22.200000
25%      39.250000
50%      44.400000
75%      50.000000
max      77.800000
Name: WomenPct, dtype: float64

Firms below 30% (critical mass threshold): 4
Firms at or above 30%: 91

Board size range: 7 to 18


In [8]:
df_clean.to_csv("data/raw/board_2025_clean.csv", index=False)
print(f"Saved {len(df_clean)} rows")

Saved 95 rows
